In [6]:
import os
import nibabel as nib
import numpy as np
import pandas as pd
from google.colab import drive

In [2]:
#Mount Google Drive to access our dataset stored there instead of uploading files repeatedly
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
#Define the base directory where our Alzheimer's disease neuroimaging dataset is stored
BASE_DIR = '/content/drive/MyDrive/Final_Dataset1'

#Define folder structure based on our dataset organization:
# - Modality types: MRI, PET, co-registered (coreg)
# - Patient classes: Alzheimer's Disease (ad) vs Control Normal (cn)
# - Data augmentation status: original vs augmented

folders = [
    'original_mri_ad', 'original_mri_cn',
    'augmented_mri_ad', 'augmented_mri_cn',
    'original_pet_ad', 'original_pet_cn',
    'augmented_pet_ad', 'augmented_pet_cn',
    'coreg_ad_augmented', 'coreg_ad_original',
    'coreg_cn_augmented', 'coreg_cn_original'
]


In [4]:
#Track file counts to verify dataset completeness and balance before model development
file_counts = {}
total_files = 0

In [5]:
print("Checking all dataset folders:")
for folder in folders:
    folder_path = os.path.join(BASE_DIR, folder)
    if not os.path.exists(folder_path):
        #Print message if there is a missing folder
        print(f"Folder {folder_path} does not exist!")
        file_counts[folder] = 0
    else:
        #Only count .nii.gz files since this is the standard format for our neuroimaging data
        files = [f for f in os.listdir(folder_path) if f.endswith('.nii.gz')]
        file_count = len(files)
        file_counts[folder] = file_count
        total_files += file_count
        print(f"Found folder: {folder_path} - Contains {file_count} files")

#Output total count to verify we have sufficient data for training
print(f"\nTotal files in dataset: {total_files}")

Checking all dataset folders:
Found folder: /content/drive/MyDrive/Final_Dataset1/original_mri_ad - Contains 72 files
Found folder: /content/drive/MyDrive/Final_Dataset1/original_mri_cn - Contains 99 files
Found folder: /content/drive/MyDrive/Final_Dataset1/augmented_mri_ad - Contains 72 files
Found folder: /content/drive/MyDrive/Final_Dataset1/augmented_mri_cn - Contains 99 files
Found folder: /content/drive/MyDrive/Final_Dataset1/original_pet_ad - Contains 72 files
Found folder: /content/drive/MyDrive/Final_Dataset1/original_pet_cn - Contains 99 files
Found folder: /content/drive/MyDrive/Final_Dataset1/augmented_pet_ad - Contains 72 files
Found folder: /content/drive/MyDrive/Final_Dataset1/augmented_pet_cn - Contains 99 files
Found folder: /content/drive/MyDrive/Final_Dataset1/coreg_ad_augmented - Contains 72 files
Found folder: /content/drive/MyDrive/Final_Dataset1/coreg_ad_original - Contains 72 files
Found folder: /content/drive/MyDrive/Final_Dataset1/coreg_cn_augmented - Contains

In [7]:
#Load neuroimaging data from NIfTI format which is the standard for brain imaging
def load_nifti_file(file_path):
    img = nib.load(file_path)
    #Convert to numpy array for analysis since we need numerical access to voxel values
    data = img.get_fdata()
    #Some scans might have time as 4th dimension; we only need spatial data for this analysis
    if len(data.shape) == 4:
        data = data[:, :, :, 0]
    return data, img.header

#Create dictionaries to store properties across different scan types for comparison
shapes = {}
file_sizes = {}
data_types = {}
sample_stats = {}

#Examine one sample from each category to understand data characteristics
for folder in folders:
    folder_path = os.path.join(BASE_DIR, folder)
    if not os.path.exists(folder_path):
        continue

    files = [f for f in os.listdir(folder_path) if f.endswith('.nii.gz')]
    if files:
        #Just take the first file as representative - detailed QC will happen later
        sample_file = os.path.join(folder_path, files[0])

        data, header = load_nifti_file(sample_file)

        #Store dimensions to verify consistency across modalities and detect any unusual sizes
        shapes[folder] = data.shape
         #Track file sizes to estimate storage requirements and potential memory constraints
        file_sizes[folder] = os.path.getsize(sample_file) / (1024 * 1024)
        #Data types impact memory usage and precision of our model calculations
        data_types[folder] = data.dtype


        #Calculate basic statistics to understand intensity distributions
        #These values help identify outliers and can inform normalization strategies
        min_val = np.min(data)
        max_val = np.max(data)
        mean_val = np.mean(data)
        std_val = np.std(data)

        sample_stats[folder] = {'min': min_val, 'max': max_val, 'mean': mean_val, 'std': std_val}

        #Display summary for each sample to quickly spot any anomalies
        print(f"\nSample from {folder}:")
        print(f"  File: {files[0]}")
        print(f"  Shape: {data.shape}")
        print(f"  File size: {file_sizes[folder]:.2f} MB")
        print(f"  Data type: {data.dtype}")
        print(f"  Value range: [{min_val:.2f}, {max_val:.2f}]")
        print(f"  Mean: {mean_val:.2f}, Std: {std_val:.2f}")

        #Voxel spacing affects the physical interpretation of our data
        #Different spacings could require resampling before combining modalities
        if header is not None:
            spacing = header.get_zooms()
            print(f"  Voxel spacing: {spacing}")

#Create a tabular summary for easy comparison across folders
shape_df = pd.DataFrame(list(shapes.items()), columns=['Folder', 'Shape'])
print(shape_df)


Sample from original_mri_ad:
  File: orig_mri_ad_64.nii.gz
  Shape: (128, 128, 128)
  File size: 2.27 MB
  Data type: float64
  Value range: [-0.36, 12.63]
  Mean: -0.00, Std: 1.00
  Voxel spacing: (np.float32(2.0), np.float32(2.0), np.float32(2.0))

Sample from original_mri_cn:
  File: orig_mri_cn_2.nii.gz
  Shape: (128, 128, 128)
  File size: 3.01 MB
  Data type: float64
  Value range: [-0.38, 9.34]
  Mean: 0.00, Std: 1.00
  Voxel spacing: (np.float32(2.0), np.float32(2.0), np.float32(2.0))

Sample from augmented_mri_ad:
  File: aug_mri_ad_19.nii.gz
  Shape: (128, 128, 128)
  File size: 4.34 MB
  Data type: float64
  Value range: [-0.40, 7.43]
  Mean: 0.03, Std: 0.97
  Voxel spacing: (np.float32(2.0), np.float32(2.0), np.float32(2.0))

Sample from augmented_mri_cn:
  File: aug_mri_cn_66.nii.gz
  Shape: (128, 128, 128)
  File size: 5.04 MB
  Data type: float64
  Value range: [-0.44, 8.33]
  Mean: 0.04, Std: 0.97
  Voxel spacing: (np.float32(2.0), np.float32(2.0), np.float32(2.0))

Sa